# Experiment 11: SGD + Momentum + Weight Decay + ReduceLROnPlateau

## Rationale

All 10 prior experiments used Adam(lr=0.001). Published FashionMNIST results typically use SGD with momentum and weight decay for better generalization.

**Single variable changed**: optimizer --- `SGD(lr=0.01, momentum=0.9, weight_decay=1e-4, nesterov=True)` + `ReduceLROnPlateau(factor=0.1, patience=5)`
**Held constant**: architecture (DiagnosticCNN), loss (CrossEntropy), data (no augmentation), epochs (40)

| Step | Description | What it does | Import path |
|------|-------------|--------------|-------------|
| 1 | Import Libraries | Load PyTorch, src modules, detect device | --- |
| 2 | Load Dataset | FashionMNIST with train/test split | `src/data_utils.py` |
| 3 | Define DiagnosticCNN | Identical architecture to E1 baseline | --- |
| 4 | Train with SGD | Train 40 epochs with SGD + Nesterov + ReduceLROnPlateau | `src/train_utils.py` |
| 5 | Evaluate Model | Per-class TPR, Precision, confusion matrix | `src/eval_utils.py` |
| 6 | ROC & PR Curves | ROC-AUC and PR-AUC scores | `src/eval_utils.py`, `src/vis_utils.py` |
| 7 | Compare with E8 | Side-by-side metrics vs Adam + CosineLR (E8) | `src/eval_utils.py` |
| 8 | Save Outputs | Save metrics to outputs/error_analysis/sgd_tuning/ | --- |

---


In [ ]:
import os, sys
# Detect project root: look for src/ directory in CWD or parents
def _find_root(marker="src", max_up=3):
    p = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if os.path.isdir(os.path.join(p, marker)):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.getcwd())
PROJ_ROOT = _find_root()
sys.path.insert(0, PROJ_ROOT)

import os, torch, torch.nn as nn, torch.optim as optim
import numpy as np
import torchvision.transforms as transforms
from src.data_utils import load_fashionmnist, get_dataloaders
from src.train_utils import train_one_epoch
from src.eval_utils import (
    evaluate_detailed, get_all_probas_and_labels,
    compute_roc_auc_scores, compute_pr_auc_scores)

OUT_DIR = "../outputs/error_analysis/sgd_tuning"
os.makedirs(OUT_DIR, exist_ok=True)

if torch.backends.mps.is_available():    device = "mps"
elif torch.cuda.is_available():          device = "cuda"
else:                                    device = "cpu"
print(f'Device: {device}')

Device: cuda


## Dataset — identical to E1

In [2]:
import sys; sys.path.append("..")

import os, torch, torch.nn as nn, torch.optim as optim
import numpy as np
import torchvision.transforms as transforms
from src.data_utils import load_fashionmnist, get_dataloaders
from src.train_utils import train_one_epoch
from src.eval_utils import (
    evaluate_detailed, get_all_probas_and_labels,
    compute_roc_auc_scores, compute_pr_auc_scores
)


Train: 938 batches, Test: 157 batches


## Architecture — DiagnosticCNN

In [3]:
**Single variable changed**: optimizer and LR schedule — `SGD(lr=0.01, momentum=0.9, weight_decay=1e-4, nesterov=True)` + `ReduceLROnPlateau(factor=0.1, patience=5)`
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1); self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1); self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1); self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1); self.bn4 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        self.conv5 = nn.Conv2d(64, 128, 3, padding=1); self.bn5 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(128, num_classes)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.relu(self.bn5(self.conv5(x)))
        x = self.pool3(x)
        x = self.gap(x).view(x.size(0), -1)
        x = self.drop(x)
        return self.fc(x)

model = DiagnosticCNN().to(device)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

Params: 140,778


c:\document\Study documents\Deeplearning_Course\.venv\Lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4, nesterov=True)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=5)
EPOCHS = 40

print(f"Optimizer: {optimizer}")
print(f"Scheduler: {scheduler}")


In [4]:
train_losses = []
model.train()
for epoch in range(EPOCHS):
    loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(loss)
    scheduler.step(loss)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {loss:.4f}")

with open(os.path.join(OUT_DIR, "train_losses.txt"), "w") as f:
    for l in train_losses:
        f.write(f"{l}\n")
torch.save(model.state_dict(), os.path.join(OUT_DIR, "model_weights.pth"))
print(f"Done. Final loss: {train_losses[-1]:.4f}")


Optimizer: SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    initial_lr: 0.01
    lr: 0.01
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0.0001
)
Scheduler: <torch.optim.lr_scheduler.StepLR object at 0x000001F2BBCC2EA0>


## Training

In [5]:
train_losses = []
model.train()
for epoch in range(EPOCHS):
    loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(loss)
    scheduler.step()
    if (epoch + 1) % 10 == 0 or epoch == 0:
        current_lr = scheduler.get_last_lr()[0]
        print(f'Epoch [{epoch+1}/{EPOCHS}] Loss: {loss:.4f}  LR: {current_lr:.6f}')

with open(os.path.join(OUT_DIR, 'train_losses.txt'), 'w') as f:
    for l in train_losses:
        f.write(f'{l}\n')
torch.save(model.state_dict(), os.path.join(OUT_DIR, 'model_weights.pth'))
print(f'Done. Final loss: {train_losses[-1]:.4f}')

Epoch [1/40] Loss: 0.4966  LR: 0.010000
Epoch [10/40] Loss: 0.1624  LR: 0.010000


KeyboardInterrupt: 

## Evaluation (bias=0)

In [ ]:
accuracy, cm, per_class = evaluate_detailed(model, test_loader, device, class_names, model_name='SGDDiagnosticCNN')
probas, labels = get_all_probas_and_labels(model, test_loader, device, 10)
roc_scores = compute_roc_auc_scores(probas, labels, model_name='SGDDiagnosticCNN')
pr_scores = compute_pr_auc_scores(probas, labels, model_name='SGDDiagnosticCNN')

cm_np = cm.cpu().numpy()
with open(os.path.join(OUT_DIR, 'metrics_summary.txt'), 'w') as f:
    f.write(f'Test Accuracy (percentage): {accuracy:.2f}\n')
    f.write(f'Test Accuracy (fraction): {accuracy / 100:.4f}\n\n')
    f.write(f'Macro ROC-AUC: {roc_scores["macro"]:.6f}\n')
    f.write(f'Macro PR-AUC:  {pr_scores["macro"]:.6f}\n\n')
    f.write(f'{"Class":<15} {"TPR":>8} {"Precision":>10}\n')
    f.write('-' * 33 + '\n')
    for i, name in enumerate(class_names):
        tpr = per_class[name]['TPR']
        prec = per_class[name]['Precision']
        f.write(f'{name:<15} {tpr:>8.4f} {prec:>10.4f}\n')

with open(os.path.join(OUT_DIR, 'confusion_matrix.txt'), 'w') as f:
    f.write(f'{"":>15}')
    for name in class_names: f.write(f'{name:>15}')
    f.write('\n')
    for i in range(10):
        f.write(f'{class_names[i]:>15}')
        for j in range(10): f.write(f'{cm_np[i, j]:>15}')
        f.write('\n')

with open(os.path.join(OUT_DIR, 'misclassification_analysis.txt'), 'w') as f:
    f.write('Misclassification Analysis\n' + '=' * 70 + '\n\n')
    for c in range(10):
        name = class_names[c]
        errors = cm_np[c].sum() - cm_np[c, c]
        f.write(f'True: {name} (errors: {errors})\n' + '-' * 50 + '\n')
        for p in np.argsort(-cm_np[c]):
            if p == c or cm_np[c, p] == 0: continue
            f.write(f'  -> {class_names[p]:<15} count={cm_np[c, p]:>4}\n')
        f.write('\n')

print(f'Baseline (bias=0) saved. Acc: {accuracy:.2f}%')
print(f'Shirt TPR: {per_class["Shirt"]["TPR"]:.4f}  Prec: {per_class["Shirt"]["Precision"]:.4f}')

## Logit bias sweep (same as E9)

In [ ]:
SHIRT_IDX = 6
BIASES = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]
sweep_results = []

model.eval()
with torch.no_grad():
    for bias in BIASES:
        all_p, all_l = [], []
        sh_tp = sh_fp = sh_fn = 0
        for inputs, lbls in test_loader:
            inputs, lbls = inputs.to(device), lbls.to(device)
            logits = model(inputs)
            logits[:, SHIRT_IDX] += bias
            preds = logits.argmax(dim=1)
            all_p.extend(preds.cpu().numpy())
            all_l.extend(lbls.cpu().numpy())
            for true, pred in zip(lbls.cpu().numpy(), preds.cpu().numpy()):
                if pred == SHIRT_IDX and true == SHIRT_IDX: sh_tp += 1
                if pred == SHIRT_IDX and true != SHIRT_IDX: sh_fp += 1
                if pred != SHIRT_IDX and true == SHIRT_IDX: sh_fn += 1
        from sklearn.metrics import accuracy_score
        acc = accuracy_score(all_l, all_p)
        sweep_results.append({'bias': bias, 'acc': round(acc*100, 2), 'tpr': round(sh_tp/(sh_tp+sh_fn+1e-8), 4), 'prec': round(sh_tp/(sh_tp+sh_fp+1e-8), 4)})
        print(f'bias={bias:+.1f}  acc={acc*100:.2f}%  Shirt TPR={sh_tp/(sh_tp+sh_fn+1e-8):.4f}  Prec={sh_tp/(sh_tp+sh_fp+1e-8):.4f}')

# find best trade-off
best_trade = max(sweep_results, key=lambda r: r['acc'] + r['tpr'] * 100)
print(f'\nBest trade-off: bias={best_trade["bias"]:+.1f}  acc={best_trade["acc"]:.2f}%  TPR={best_trade["tpr"]:.4f}  Prec={best_trade["prec"]:.4f}')

with open(os.path.join(OUT_DIR, 'bias_sweep_results.txt'), 'w') as f:
    header = f'{"Bias":>6} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}'
    f.write(header + '\n' + '-' * len(header) + '\n')
    for r in sweep_results:
        f.write(f'{r["bias"]:>+5.1f} {r["acc"]:>7.2f} {r["tpr"]:>9.4f} {r["prec"]:>10.4f}\n')
    f.write(f'\nBest trade-off: bias={best_trade["bias"]:+.1f}  acc={best_trade["acc"]:.2f}%  TPR={best_trade["tpr"]:.4f}  Prec={best_trade["prec"]:.4f}\n')

## Comparison: E11 vs E9 (Adam) best trade-off

In [ ]:
print(f'{"Optimizer":<15} {"Bias":>5} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}')
print('-' * 46)
print(f'{"E9 Adam":<15} {"+1.0":>5} {"93.17":>7} {"0.8110":>9} {"0.7776":>10}')
bt = best_trade
print(f'{"E11 SGD":<15} {bt["bias"]:>+5.1f} {bt["acc"]:>7.2f} {bt["tpr"]:>9.4f} {bt["prec"]:>10.4f}')